In [2]:
# Yelp suitability check (self-contained)
import json
from pathlib import Path
import pandas as pd
import h3

# --- 1) Locate or download the data -------------------------------------------------
# Option A: set the folder manually if you already downloaded + unzipped the Kaggle archive.
YELP_DIR = None   # e.g. Path("data/yelp")

if YELP_DIR is None:
    import kagglehub   # needs Kaggle credentials (KAGGLE_USERNAME / KAGGLE_KEY or ~/.kaggle/kaggle.json)
    YELP_DIR = Path(kagglehub.dataset_download("yelp-dataset/yelp-dataset"))
YELP_DIR = Path(YELP_DIR)

def find(name):
    hits = list(YELP_DIR.rglob(name))
    assert hits, f"{name} not found under {YELP_DIR}"
    return hits[0]

BUSINESS = find("yelp_academic_dataset_business.json")
TIPS = find("yelp_academic_dataset_tip.json")
print(BUSINESS, TIPS, sep="\n")

# --- 2) Load only needed columns -----------------------------------------------------
biz = pd.read_json(BUSINESS, lines=True)[
    ["business_id", "name", "city", "state", "latitude", "longitude", "review_count"]]
tips = pd.read_json(TIPS, lines=True)[["business_id", "user_id", "text", "date"]]
print("businesses:", len(biz), "| tips:", len(tips))

yt = tips.merge(biz.rename(columns={"latitude": "lat", "longitude": "lon"}),
                on="business_id", how="inner")
print("tips with business coordinates:", len(yt), f"({len(yt)/len(tips):.1%})")
print("missing coordinates:", yt[["lat", "lon"]].isna().sum().to_dict())
print("distinct coordinate pairs:", yt[["lat", "lon"]].drop_duplicates().shape[0])

# --- 3) Metro areas = H3 res-4 cells (the 'city' field is messy) ---------------------
yt["h3_r4"] = [h3.latlng_to_cell(a, b, 4) for a, b in zip(yt["lat"], yt["lon"])]
metro = (yt.groupby("h3_r4")
           .agg(n_tips=("text", "size"),
                n_biz=("business_id", "nunique"),
                n_users=("user_id", "nunique"),
                city=("city", lambda s: s.value_counts().index[0]),
                state=("state", lambda s: s.value_counts().index[0]))
           .sort_values("n_tips", ascending=False))
print(metro.head(15))
print("metros with 20k+ tips:", (metro["n_tips"] >= 20000).sum())

# --- 4) Concentration and grid density in the top-3 metros ---------------------------
for cell in metro.index[:3]:
    m = yt[yt["h3_r4"] == cell]
    per_biz = m.groupby("business_id").size().sort_values(ascending=False)
    print(f"\n{metro.loc[cell, 'city']}: {len(m)} tips")
    print("  tips per business: median", per_biz.median(), "| max", per_biz.max(),
          "| top-10 share", f"{per_biz.head(10).sum()/len(m):.1%}",
          "| businesses with 10+ tips", f"{(per_biz >= 10).mean():.1%}")
    for r in (8, 9):
        cells = pd.Series([h3.latlng_to_cell(a, b, r) for a, b in zip(m["lat"], m["lon"])]).value_counts()
        print(f"  res {r}: cells={len(cells)}, median={cells.median():.0f}, share 10+={(cells >= 10).mean():.1%}")

# --- 5) Text quality -----------------------------------------------------------------
yt["len"] = yt["text"].str.len()
print("\ntip length quantiles:", yt["len"].quantile([.1, .5, .9]).to_dict())
print("share < 15 chars:", round((yt["len"] < 15).mean(), 3))
print("exact duplicate texts:", round(yt["text"].str.lower().duplicated().mean(), 3))

/home/roman/Documents/university/thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 4.07G/4.07G [02:43<00:00, 26.7MB/s]

Extracting files...


/home/roman/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4/yelp_academic_dataset_business.json
/home/roman/.cache/kagglehub/datasets/yelp-dataset/yelp-dataset/versions/4/yelp_academic_dataset_tip.json
businesses: 150346 | tips: 908915
tips with business coordinates: 908915 (100.0%)
missing coordinates: {'lat': 0, 'lon': 0}
distinct coordinate pairs: 99813
                 n_tips  n_biz  n_users             city state
h3_r4                                                         
842a135ffffffff  158144  17880    54501     Philadelphia    PA
8444465ffffffff  105997   7735    40039      New Orleans    LA
84441a9ffffffff   91558  10042    37887            Tampa    FL
84264d1ffffffff   61255   6317    25419        Nashville    TN
842661bffffffff   55824   6067    17270     Indianapolis    IN
8448e85ffffffff   55650   6009    17499           Tucson    AZ
8426463ffffffff   53170   5954    18293      Saint Louis    MO
8429891ffffffff   53016   5286    18532             Reno   